# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [ ]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [ ]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [ ]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6


## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [ ]:
# TODO: your first tool, decorated with @function_tool


## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [ ]:
# TODO: your second tool, decorated with @function_tool


## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [ ]:
# TODO: your third tool, decorated with @function_tool


## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [ ]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [ ]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [ ]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`?
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.